In [ ]:
import pandas as pd
import numpy as np

# Первичная обработка

In [ ]:
data_2018 = pd.read_csv("f1_data_2018_full.csv")
data_2019 = pd.read_csv("f1_data_2019_full.csv")
data_2020 = pd.read_csv("f1_data_2020_full.csv")
data_2021 = pd.read_csv("f1_data_2021_full.csv")
data_2022 = pd.read_csv("f1_data_2022_full.csv")
data_2023 = pd.read_csv("f1_data_2023_full.csv")
data_2024 = pd.read_csv("f1_data_2024_full.csv")
data_2025 = pd.read_csv("f1_data_2025_full.csv")

In [ ]:
data = pd.concat([data_2018, data_2019, data_2020, data_2021, data_2022, data_2023, data_2024, data_2025], ignore_index=True)

Ниже проходит смена названий команд. В случае, если одна команда переходит к другому владельцу или просто по каким-то причинам меняет название, возможны два исхода: команда остаётся с теми же инженерами, гонщиками, руководителями, спортивной базой или команда кардинально меняется. Если это первый случай, то все названия команды с 2018 по 2025 можно объединить в одно - на 2025 год - для сохранения данных о команде. Кроме как кода ниже, подобная работа была частично проведена в предыдущем ноутбуке.

In [ ]:
data['Team'] = data['Team'].apply(
    lambda x: 'Racing Bulls' if x == 'Toro Rosso' else x
)
data['Engine'] = data.apply(
    lambda x: 'Honda' if x['Team'] == 'Racing Bulls' else x['Engine'], axis=1
)

Добавляем информацию о незаполненной практике 2019 года в Австралии

In [ ]:
practice_data = {
    'Year': 2019,
    'Round': 1,
    'Place': 'Melbourne',
    'FP1': [
        ('HAM', 1, '1:23.599'), ('VET', 2, '1:23.637'), ('LEC', 3, '1:23.673'),
        ('VER', 4, '1:23.792'), ('BOT', 5, '1:23.866'), ('RAI', 6, '1:24.816'),
        ('KVY', 7, '1:24.832'), ('GAS', 8, '1:24.932'), ('MAG', 9, '1:24.934'),
        ('HUL', 10, '1:25.015'), ('GIO', 11, '1:25.166'), ('GRO', 12, '1:25.224'),
        ('ALB', 13, '1:25.230'), ('SAI', 14, '1:25.285'), ('STR', 15, '1:25.288'),
        ('PER', 16, '1:25.498'), ('RIC', 17, '1:25.634'), ('NOR', 18, '1:25.966'),
        ('KUB', 19, '1:27.914'), ('RUS', 20, '1:28.740')
    ],
    'FP2': [
        ('HAM', 1, '1:22.600'), ('BOT', 2, '1:22.648'), ('VER', 3, '1:23.400'),
        ('GAS', 4, '1:23.442'), ('VET', 5, '1:23.473'), ('RAI', 6, '1:23.572'),
        ('HUL', 7, '1:23.574'), ('RIC', 8, '1:23.574'), ('LEC', 9, '1:23.754'),
        ('GRO', 10, '1:23.814'), ('KVY', 11, '1:23.933'), ('MAG', 12, '1:23.988'),
        ('STR', 13, '1:24.011'), ('SAI', 14, '1:24.133'), ('GIO', 15, '1:24.293'),
        ('PER', 16, '1:24.401'), ('ALB', 17, '1:24.675'), ('NOR', 18, '1:24.733'),
        ('RUS', 19, '1:26.453'), ('KUB', 20, '1:26.655')
    ],
    'FP3': [
        ('HAM', 1, '1:22.292'), ('VET', 2, '1:22.556'), ('LEC', 3, '1:22.749'),
        ('GRO', 4, '1:23.112'), ('MAG', 5, '1:23.334'), ('GAS', 6, '1:23.367'),
        ('BOT', 7, '1:23.422'), ('KVY', 8, '1:23.442'), ('VER', 9, '1:23.481'),
        ('RIC', 10, '1:23.695'), ('HUL', 11, '1:23.737'), ('GIO', 12, '1:23.831'),
        ('SAI', 13, '1:24.049'), ('PER', 14, '1:24.082'), ('ALB', 15, '1:24.328'),
        ('STR', 16, '1:24.345'), ('RAI', 17, '1:24.402'), ('NOR', 18, '1:24.568'),
        ('RUS', 19, '1:25.944'), ('KUB', 20, '1:26.689')
    ]
}

def get_practice_positions(practice_list):
    return {driver: pos for driver, pos, _ in practice_list}

fp1_positions = get_practice_positions(practice_data['FP1'])
fp2_positions = get_practice_positions(practice_data['FP2'])
fp3_positions = get_practice_positions(practice_data['FP3'])

In [ ]:
australia_mask = (data['Year'] == 2019) & (data['Place'] == 'Melbourne')
data.loc[australia_mask, 'P1'] = data.loc[australia_mask, 'Driver'].map(fp1_positions)
data.loc[australia_mask, 'P2'] = data.loc[australia_mask, 'Driver'].map(fp2_positions)
data.loc[australia_mask, 'P3'] = data.loc[australia_mask, 'Driver'].map(fp3_positions)
data['P1'] = data['P1'].fillna(0).astype(int)
data['P2'] = data['P2'].fillna(0).astype(int)
data['P3'] = data['P3'].fillna(0).astype(int)

Добавляем пропущенные значения в признак Strengh_of_first_partners - для каждого гонщика оцениваем силу его партнёра по команде в первые годы гонок.

In [ ]:
data['Strengh_of_first_partners'] = data.apply(
    lambda x: 7 if x['Driver'] == 'VER' or x['Driver'] == 'STR' else x['Strengh_of_first_partners'], axis = 1
)
data['Strengh_of_first_partners'] = data.apply(
    lambda x: 5 if x['Driver'] == 'KUB' else x['Strengh_of_first_partners'], axis = 1
)

Добавление пропущенных данных о зарплатах

In [ ]:
data['Salary'] = data.apply(
    lambda x: 10 if x['Driver'] == 'STR' and x['Year'] == 2022 else x['Salary'], axis = 1
)
data['Salary'] = data.apply(
    lambda x: 2 if x['Driver'] == 'RIC' and x['Year'] == 2023 else x['Salary'], axis = 1
)
data['Salary'] = data.apply(
    lambda x: 1 if x['Driver'] == 'DEV' and x['Year'] == 2022 else x['Salary'], axis = 1
)
data['Salary'] = data.apply(
    lambda x: 1 if x['Driver'] == 'COL' and x['Year'] == 2024 else x['Salary'], axis = 1
)
data['Salary'] = data.apply(
    lambda x: 2 if x['Driver'] == 'LAW' and x['Year'] == 2024 else x['Salary'], axis = 1
)
data['Salary'] = data.apply(
    lambda x: 1 if x['Driver'] == 'BEA' and x['Year'] == 2024 else x['Salary'], axis = 1
)
data['Salary'] = data.apply(
    lambda x: 1 if x['Driver'] == 'LAW' and x['Year'] == 2023 else x['Salary'], axis = 1
)
data['Salary'] = data.apply(
    lambda x: 2 if x['Driver'] == 'HUL' and x['Year'] == 2020 else x['Salary'], axis = 1
)
data['Salary'] = data.apply(
    lambda x: 0.2 if x['Driver'] == 'AIT' and x['Year'] == 2020 else x['Salary'], axis = 1
)
missing_salaries = {
    (2020, 'FIT'): 0.2,
    (2021, 'KUB'): 0.15,
    (2024, 'DOO'): 0.5,
    (2025, 'BOR'): 2.0,
    (2025, 'HAD'): 0.5,
    (2025, 'COL'): 0.75,
}
def fill_missing_salaries(row):
    key = (row['Year'], row['Driver'])
    return missing_salaries.get(key, row['Salary'])
data['Salary'] = data.apply(fill_missing_salaries, axis=1)

In [ ]:
data.to_csv("data.csv", index=False)

In [ ]:
missing = pd.DataFrame({
    'Колонка': data.columns,
    'Пропусков': data.isna().sum().values,
    'Доля_%': (data.isna().sum() / len(data) * 100).values
}).sort_values('Пропусков', ascending=False)


In [ ]:
data['Rainfall'] = data['Rainfall'].map({'True': 1, 'False': 0})

# Обработка признаков

In [ ]:
from sklearn.preprocessing import TargetEncoder

In [ ]:
data = pd.read_csv('data.csv')

Target Encoding фамилии гонщика, команды и национальности

In [ ]:
data['Driver'] = data['Driver'].str.strip()
data['Driver'] = data['Driver'].str.upper()
import re
data['Driver'] = data['Driver'].apply(lambda x: re.sub(r'[^A-Z]', '', str(x)))

In [ ]:
original_drivers = data['Driver'].copy()
enc = TargetEncoder(smooth="auto")
X = original_drivers.values.reshape(-1, 1)
y = data['Result'].values
data['Driver_encoded'] = enc.fit_transform(X, y)

Сделаем словарь соответствий гонщиков и их закодированных номеров для дальнейших предсказаний.

In [ ]:
part_data = data[(data['Year'] == 2025) | (data['Year'] == 2024)]
part_data = part_data[['Driver', 'Driver_encoded']].drop_duplicates(subset=['Driver'])
drivers = dict(zip(part_data['Driver'], part_data['Driver_encoded']))

In [ ]:
import json

In [ ]:
with open('drivers.json', 'w', encoding='utf-8') as f:
    json.dump(drivers, f, ensure_ascii=False, indent=2)

Добавляем национальность водителей

In [ ]:
data['Driver_nationalitie'] = data['Driver_nationalitie'].str.strip()
data['Driver_nationalitie'] = data['Driver_nationalitie'].str.upper()
import re
data['Driver_nationalitie'] = data['Driver_nationalitie'].apply(lambda x: re.sub(r'[^A-Z]', '', str(x)))

In [ ]:
original_nationalities = data['Driver_nationalitie'].copy()
X = original_nationalities.values.reshape(-1, 1)
Y = data['Result'].copy()
data['Encoded_driver_nations'] = enc.fit_transform(X, Y)

In [ ]:
from sklearn.preprocessing import TargetEncoder

enc_team = TargetEncoder(smooth="auto")
data['Team_encoded'] = enc_team.fit_transform(
    data['Team'].values.reshape(-1, 1),
    data['Result']
)
part_teams = data[(data['Year'] == 2025) | (data['Year'] == 2024)]
part_teams = part_teams[['Team', 'Team_encoded']].drop_duplicates(subset=['Team'])
teams_dict = dict(zip(part_teams['Team'], part_teams['Team_encoded']))
teams_dict['Cadillac'] = round(
    (teams_dict['Alpine'] + teams_dict['Haas F1 Team'] + teams_dict['Aston Martin']) / 3
)
teams_dict['Audi'] = teams_dict['Sauber']  # или Sauber, если так называется

print(f"Найдено команд: {len(teams_dict)}")
print(teams_dict)

Найдено команд: 12
{'Williams': 17.45878127969819, 'Aston Martin': 13.35315943011907, 'Sauber': 16.528063182926513, 'Alpine': 13.730556380871556, 'Mercedes': 6.7342288331709055, 'Haas F1 Team': 17.433398436435873, 'Ferrari': 8.978690955828185, 'McLaren': 10.536791617707653, 'Red Bull Racing': 8.51974271610397, 'Racing Bulls': 15.93629674616412, 'Cadillac': 15, 'Audi': 16.528063182926513}


In [ ]:
with open('teams_encoded.json', 'w', encoding='utf-8') as f:
    json.dump(teams_dict, f, ensure_ascii=False, indent=2)

In [ ]:
data.to_csv('data.csv', index=False)

Кодируем производителей моторов с помощью One hot Encoder

In [ ]:
engines = pd.get_dummies(data['Engine'], prefix='Engine', dtype=int)

In [ ]:
data = pd.concat([data, engines], axis = 1)

Кодируем места проведения гонок с помощью LabelEncoder()

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()
data['Place_encoded'] = le.fit_transform(data['Place'])
unique_places = data[['Place', 'Place_encoded']].drop_duplicates(subset=['Place'])
places = dict(zip(unique_places['Place'], unique_places['Place_encoded']))
places = {k: int(v) for k, v in places.items()}
print(places)

{'Melbourne': 12, 'Sakhir': 23, 'Shanghai': 25, 'Baku': 1, 'Barcelona': 2, 'Monte Carlo': 17, 'Montréal': 18, 'Le Castellet': 9, 'Spielberg': 30, 'Silverstone': 26, 'Hockenheim': 4, 'Budapest': 3, 'Spa-Francorchamps': 29, 'Monza': 19, 'Singapore': 27, 'Sochi': 28, 'Suzuka': 31, 'Austin': 0, 'Mexico City': 13, 'São Paulo': 32, 'Yas Marina': 34, 'Yas Island': 33, 'Mugello': 20, 'Nürburgring': 21, 'Portimão': 22, 'Imola': 5, 'Istanbul': 6, 'Zandvoort': 35, 'Lusail': 10, 'Jeddah': 7, 'Miami': 14, 'Monaco': 16, 'Marina Bay': 11, 'Las Vegas': 8, 'Miami Gardens': 15, 'Sao Paulo': 24}


In [ ]:
with open('places.json', 'w', encoding='utf-8') as f:
    json.dump(places, f, ensure_ascii=False, indent=2)

Обработаем некоторые числовые значения

In [ ]:
nan_count = data['Age'].isna().sum()
print(nan_count)
nan_count = data['Salary'].isna().sum()
print(nan_count)

0
0


In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
data['Salary_log'] = np.log1p(data['Salary'])
data['Salary_encoded'] = scaler.fit_transform(data[['Salary_log']])
del data['Salary_log']

In [ ]:
data.to_csv('data.csv', index=False)

In [ ]:
track_features = ['CircleLength', 'Quantity_of_circles', 'Quantity_of_drs',
                  'Quantity_of_turns', 'Tire_wear', 'Speed_of_turns',
                  'Downforce', 'Brake_load', 'Speed_of_track', 'PitStop_losing_time']

scaler_track = StandardScaler()
data[track_features] = scaler_track.fit_transform(data[track_features])

In [ ]:
weather_cols = ['AirTemp', 'TrackTemp', 'Humidity', 'WindSpeed', 'Pressure']

for col in weather_cols:
    data[col] = data.groupby('Place')[col].transform(lambda x: x.fillna(x.mean()))

scaler_weather = StandardScaler()
data[weather_cols] = scaler_weather.fit_transform(data[weather_cols])

In [ ]:
data['Rainfall'] = data['Rainfall'].map({'True': 1, 'False': 0})

In [ ]:
data.to_csv('final_data.csv', index=False)
data.to_csv('data.csv', index=False)

Ниже работа со словарями - при предсказании возникли проблемы что в зависимости от среды, в которой выполняется код, коды водителей и команд могут округляться, поэтому делаем их одинаковыми

In [ ]:
drivers_d = {
  "ALB": 14.053,
  "ALO": 12.832,
  "BOT": 11.586,
  "GAS": 15.106,
  "HAM": 6.2921,
  "HUL": 15.617,
  "LEC": 10.111,
  "MAG": 17.197,
  "NOR": 8.230,
  "OCO": 14.985,
  "PER": 10.535,
  "PIA": 8.828,
  "RIC": 13.430,
  "RUS": 11.709,
  "SAI": 11.334,
  "SAR": 19.219,
  "STR": 14.075,
  "TSU": 16.919,
  "VER": 7.093,
  "ZHO": 17.142,
  "BEA": 12.424,
  "COL": 18.266,
  "LAW": 14.993,
  "DOO": 19.709,
  "ANT": 11.443,
  "BOR": 21.044,
  "HAD": 12.323,
  "LIN": 16.5
}

In [ ]:
data['Driver_encoded'] = data['Driver'].map(drivers_d)

In [ ]:
data['Team'].unique()

array(['McLaren', 'Mercedes', 'Sauber', 'Racing Bulls', 'Haas F1 Team',
       'Renault', 'Racing Point', 'Ferrari', 'Red Bull Racing',
       'Williams', 'Alpine', 'Aston Martin'], dtype=object)

In [ ]:
teams_d = {
  "Williams": 17.683,
  "Aston Martin": 13.668,
  "Sauber": 16.278,
  "Alpine": 14.663,
  "Mercedes": 6.907,
  "Haas F1 Team": 17.266,
  "Ferrari": 10.303,
  "McLaren": 11.176,
  "Red Bull Racing": 9.077,
  "Racing Bulls": 15.758,
  "Cadillac": 15,
  "Audi": 16.278
}

In [ ]:
data['Team_encoded'] = data['Team'].map(teams_d)

In [ ]:
original_columns_to_remove = [
    'Team',
    'Place',
    'Engine',
    'Nationality',
    'Driver_nationalitie',
    'Salary_log',
    'Team_Year',
]
columns_to_drop = [col for col in original_columns_to_remove if col in data.columns]
data = data.drop(columns=columns_to_drop)

In [ ]:
data.to_csv('data.csv', index=False)
data.to_csv('final_data_fixed.csv', index=False)